# Knowledge Distillation

Using agentic AI, this notebook builds and distills knowledge for Botanical Gardens. This can easily be expanded to other domains.

In [ ]:
OPENAI_API_KEY = "sk-..."

import openai

client = openai.Client(api_key=OPENAI_API_KEY)


responses = client.chat.completions.create(
    model="gpt-5.1",
    messages=[
        {"role": "user", "content": "Oh Holy OpenAI, we humbly as to be blessed that this notebook succeeds. Please write a quick poem as a blessing."}
    ]
)

print(responses.choices[0].message.content)

In [ ]:
def get_prompt(location):
    return f"""You are a helpful assistant who writes a detailed overview of specific botanical garden locations around the world. Please include the following sections in your overview.

1. Summary
2. Location and surrounding metropolitan area.
    a. Cities nearby
    b. Estimated population within 100 miles
3. Attractions within the garden
4. History of the garden
5. Fundraisers and Endowments

Please provide the information in the following XML format:
<summary>
</summary>
<location>
</location>
<attractions>
</attractions>
<history>
</history>
<fundraisers>
</fundraisers>

Please write several paragaphs for each section and include all the detail you can. Please also use web search to find the most up-to-date information about the location. No need to include citations.
"""

def generate_botanical_garden_overview(location):
    prompt = get_prompt(location)
    response = client.responses.create(
        model="gpt-5.1",
        input=prompt,
        tools=[{"type": "web_search"}],
    )
    return response.output[-1].content[-1].text

location = "Kew Gardens, London"
res = generate_botanical_garden_overview(location)
# print(overview)

In [ ]:
print(res)

In [ ]:
import os
from tqdm import tqdm

os.makedirs("data", exist_ok=True)

BOTANICAL_GARDENS = [
    "Mitchell Park Domes, Milwaukee",
    "Green Bay Botanical Garden, Green Bay",
    "Olbrich Botanical Gardens, Madison",
    "Lincoln Park Conservatory, Chicago",
    "Garfield Park Conservatory, Chicago",
    "Nicolas Conservatory and Gardens, Rockford",
    "Brooklyn Botanic Garden, Brooklyn",
    "Longwood Gardens, Kennett Square",
    "New York Botanical Garden, Bronx",
    "Missouri Botanical Garden, St. Louis",
    "The Huntingon, San Marino",
    "Desert Botanical Garden, Phoenix",
    "Atlanta Botanical Garden, Atlanta",
    "Franklin Park Conservatory and Botanical Gardens, Columbus",
    "Chihuly Garden and Glass, Seattle",
    "Red Butte Garden and Arboretum, Salt Lake City",
    "Foellinger Botanical Conservatory, Fort Wayne",
    "Daniel Stowe Botanical Garden, Belmont",
    "The Gardens of Golden Gate Park, San Francisco",
    "Phipps Conservatory and Botanical Gardens, Pittsburgh",
    "United States Botanic Garden, Washington D.C."
]

for garden in tqdm(BOTANICAL_GARDENS):
    file_name = garden.lower().replace(" ", "_").replace(",", "") + ".txt"
    file_path = os.path.join("data", file_name)
    if not os.path.exists(file_path):
        overview = generate_botanical_garden_overview(garden)
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(overview)
        print(f"Saved overview for {garden} to {file_path}")

In [ ]:
from glob import glob

CONTEXT = ""

for file in glob("./data/*.txt"):
    CONTEXT += open(file, "r", encoding="utf-8").read() + "\n\n"

# The file the API reads
with open("../chat_context.txt", "w", encoding="utf-8") as f:
    f.write(CONTEXT)